# Exp 1: Critic-only vs Both weighting ablation

Reads `progress.csv` from rlkit log directories and computes normalized return mean ± std.  
Outputs a comparison table and saves `results/exp1_critic_vs_both.csv`.

**Prerequisites:** run the train scripts first:
```bash
bash train_scripts/ablation_critic_both/train_hopper.sh
bash train_scripts/ablation_critic_both/train_walker.sh
bash train_scripts/ablation_critic_both/train_halfcheetah.sh
```

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gym
import d4rl  # noqa

# Run from project root
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
print('Working dir:', os.getcwd())

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
ENVS = ['hopper-medium-v2', 'walker2d-medium-v2', 'halfcheetah-medium-v2']

VARIANTS = {
    'critic_only': 'saves_ablation_critic_only_identity',
    'both':        'saves_ablation_both_identity',
}

N_SEEDS  = 5
EVAL_COL = 'evaluation/Average Returns'   # rlkit default column

In [ ]:
def get_normalized_score(env_name, raw_return):
    env = gym.make(env_name)
    score = env.get_normalized_score(raw_return) * 100
    env.close()
    return score

def read_final_return(root, env, seed):
    """Return the last logged Average Returns for a given seed."""
    pattern = os.path.join(root, '**', f'seed_{seed}', '**', env, '**', 'progress.csv')
    files = glob.glob(pattern, recursive=True)
    if not files:
        return None
    df = pd.read_csv(files[0])
    if EVAL_COL not in df.columns:
        print(f'[WARN] Column not found in {files[0]}')
        return None
    return df[EVAL_COL].dropna().iloc[-1]

In [ ]:
rows = []
for variant, save_root in VARIANTS.items():
    for env in ENVS:
        raw_returns = []
        for seed in range(N_SEEDS):
            r = read_final_return(save_root, env, seed)
            if r is not None:
                raw_returns.append(r)
            else:
                print(f'[WARN] Missing: variant={variant}, env={env}, seed={seed}')

        norm = [get_normalized_score(env, r) for r in raw_returns]
        rows.append({
            'variant': variant,
            'env':     env,
            'n_seeds': len(norm),
            'mean':    round(np.mean(norm), 2) if norm else float('nan'),
            'std':     round(np.std(norm),  2) if norm else float('nan'),
        })

df = pd.DataFrame(rows)
df

In [ ]:
# Pivot: rows = env, columns = variant
pivot = df.pivot(index='env', columns='variant', values=['mean', 'std'])
pivot.columns = [f'{v}_{stat}' for stat, v in pivot.columns]

for variant in VARIANTS:
    pivot[variant] = (pivot[f'{variant}_mean'].map('{:.1f}'.format)
                      + ' ± '
                      + pivot[f'{variant}_std'].map('{:.1f}'.format))

display(pivot[list(VARIANTS.keys())])

In [ ]:
fig, axes = plt.subplots(1, len(ENVS), figsize=(12, 4), sharey=False)

for ax, env in zip(axes, ENVS):
    env_df = df[df['env'] == env]
    bars = ax.bar(env_df['variant'], env_df['mean'],
                  yerr=env_df['std'], capsize=6,
                  color=['steelblue', 'tomato'], alpha=0.8, width=0.5)
    ax.set_title(env, fontsize=11)
    ax.set_ylabel('Normalized Return (%)')
    ax.set_xlabel('')
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('Critic-only vs Both weighting (5 seeds)', fontsize=13, y=1.02)
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/exp1_critic_vs_both.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
out = 'results/exp1_critic_vs_both.csv'
df.to_csv(out, index=False)
print(f'Saved to {out}')